# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ashishpal003/flyrank_ml_intern/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Binary classification → calibrated probability → ranking.** (Not multiclass, not clustering, not regression on the raw traffic number.)

- The editor's decision is "shortlist this page this cycle, or not" — a yes/no. But it is made under a **fixed review budget**, so the useful output is a score that *orders* pages by risk; how many to actually flag (the operating point K) is chosen separately.
- We predict a **probability**, not a hard label: misses cost more than false alarms, and the editor wants a confidence-ranked queue with reason codes, not a binary dump.
- **Regression on "next-30d impressions" is rejected:** the target is heavy-tailed and dominated by a few big pages, and the decision only needs "materially down vs not," not a point forecast.

Against the `framing-ml-problems` table this is the **"which ones first?" → ranking/scoring** row, which fixes the metric in section 3 (precision@K).

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

CSV = Path("../../data/raw/content_refresh_anonymized.csv")
if not CSV.exists():
    CSV = Path("data/raw/content_refresh_anonymized.csv")  # when run from repo root
df = pd.read_csv(CSV)

print(f"rows:              {len(df):,}")
print(f"unique content_id: {df['content_id'].nunique():,}  -> one row = one pseudonymized page: {df['content_id'].nunique() == len(df)}")
print(f"unique client_id:  {df['client_id'].nunique()}  (enough for a client-grouped split)")
print(f"columns:           {df.shape[1]}")

rows:              30,000
unique content_id: 30,000  -> one row = one pseudonymized page: True
unique client_id:  32  (enough for a client-grouped split)
columns:           44


## 2. Target or proxy

**What we would predict:** `P(sustained impressions decline over the label window)` for an eligible, currently-healthy page.

**Where the real label comes from — an OBSERVED outcome, not a defined rule.** On the warehouse (ML-04) it is measured from daily impressions in `[decision_date, decision_date + 30d]`: **1** if label-window impressions run ≈ 25%+ below the trailing 90-day pace *and* the drop persists (not a one-day spike), on pages above a ~100-impressions / prior-30d floor. The feature window ends *before* `decision_date`, so nothing in the features can see the label.

**The label trap (stated so we never fall in it).** In the starter data `is_declining_label` = `(trend_direction == "down")`, and `trend_direction` / `trend_pct` are a *rule applied to the very same 30-day windows the features are built from*. A model fed those columns learns the rule, not the world. They are on the **leakage red list** — never features — and they are **not** our target.

**Starter-CSV proxy (this notebook only).** The starter slice has no forward window, so section 4's dataframe uses a stand-in built from the columns that *do* exist:

```
proxy_decline = impressions_last_30d < (1 - 0.25) * impressions_prev_30d
```

This is directional illustration only. It unavoidably reuses the same last-30 / prev-30 comparison that `trend_direction` uses — which is *exactly* the limitation the ML-04 forward label fixes. The cell below shows `proxy_decline` sits *inside* `trend_direction == "down"` (stricter threshold), and that ~800 "down" pages fall in the −20%..−25% band the proxy leaves out.

In [2]:
# Eligibility (see section 4 for the reasoning) — needed before the proxy base rate is meaningful.
elig = (
    (df["impressions_prev_30d"] >= 100)
    & (df["content_age_days"] >= 90)
    & ((df["avg_position"] > 0) | (df["impressions_90d"] > 0))
)
freefall = (df["trend_direction"] == "down") & (df["trend_pct"] <= -50)
e = df[elig & ~freefall].copy()

e["proxy_decline"] = (e["impressions_last_30d"] < 0.75 * e["impressions_prev_30d"]).astype(int)

base_rate = e["proxy_decline"].mean()
down = e["trend_direction"] == "down"
print(f"eligible pages:              {len(e):,}")
print(f"proxy_decline base rate:     {base_rate:.3f}")
print(f"proxy=1 AND trend='down':    {int(((e['proxy_decline'] == 1) & down).sum()):,}")
print(f"proxy=1 AND trend!='down':   {int(((e['proxy_decline'] == 1) & ~down).sum()):,}  (proxy is a strict subset of 'down')")
print(f"trend='down' AND proxy=0:    {int((down & (e['proxy_decline'] == 0)).sum()):,}  (the -20%..-25% band the proxy drops)")

eligible pages:              12,304
proxy_decline base rate:     0.367
proxy=1 AND trend='down':    4,518
proxy=1 AND trend!='down':   0  (proxy is a strict subset of 'down')
trend='down' AND proxy=0:    866  (the -20%..-25% band the proxy drops)


## 3. Success metric

**Primary metric: precision@K**, where K is the editor's review budget for a cycle. Reported *alongside* it, always: **recall@K**, **average precision**, and the **base rate** (a bare "precision" with no base rate beside it is banned by the honest-claims skill).

**What "good" means — a relative bar we can defend today:**

1. precision@K **clearly above the eligible-page base rate** — the ranking beats picking pages at random; and
2. precision@K **above FlyRank's stale-visible rule** (`days_since_last_update >= 180 AND impressions_90d >= 500`) scored at the same K, on the same eligible pages.
3. Secondary, only if it holds up out-of-sample: median **lead time** vs the standard 30-day trend bucket > 0.

**No absolute target** ("precision@50 ≥ 0.40") is committed until there is an honest out-of-sample number in ML-08. Choosing the bar after seeing the model is the exact failure `framing-ml-problems` warns against.

**Why precision@K, not ROC-AUC or accuracy:** review capacity is fixed, so only the top of the ranking is ever acted on. AUC rewards separation across the whole score range we never use; accuracy is meaningless against an imbalanced forward base rate.

The cell below shows the metric is **computable today on a baseline** (the framing skill's verification step): a transparent, non-circular heuristic — *rank by how much clicks fell, last-30 vs prev-30* — scored against `proxy_decline`, next to the stale-visible rule.

In [3]:
def precision_recall_at_k(labels, scores, k):
    order = np.argsort(-np.asarray(scores, dtype=float), kind="stable")
    top = np.asarray(labels)[order[:k]]
    total_pos = np.asarray(labels).sum()
    return top.mean(), top.sum() / total_pos


def average_precision(labels, scores):
    order = np.argsort(-np.asarray(scores, dtype=float), kind="stable")
    y = np.asarray(labels)[order]
    cum_tp = np.cumsum(y)
    precision_at_i = cum_tp / (np.arange(len(y)) + 1)
    return (precision_at_i * y).sum() / y.sum()


y = e["proxy_decline"].to_numpy()
# Transparent baseline score: bigger = clicks fell more (NOT built from the impressions ratio the label uses).
clicks_drop_score = -(e["clicks_last_30d"] + 1) / (e["clicks_prev_30d"] + 1)
stale_visible = ((e["days_since_last_update"] >= 180) & (e["impressions_90d"] >= 500)).to_numpy()

print(f"base rate (random pick):     {y.mean():.3f}")
print(f"average precision (clicks-drop rank): {average_precision(y, clicks_drop_score):.3f}\n")
for k in (50, int(round(0.05 * len(e)))):
    p, r = precision_recall_at_k(y, clicks_drop_score, k)
    print(f"K={k:>4}  clicks-drop rank   precision@K={p:.3f}  recall@K={r:.3f}")
print()
n_fires = int(stale_visible.sum())
print(f"stale-visible rule fires on only {n_fires} of {len(e):,} eligible pages")
print(f"           precision among those {n_fires}: {y[stale_visible].mean():.3f}  "
      f"(recall {y[stale_visible].sum() / y.sum():.3f}) -- blind to almost all of the risk")

base rate (random pick):     0.367
average precision (clicks-drop rank): 0.444

K=  50  clicks-drop rank   precision@K=0.760  recall@K=0.008
K= 615  clicks-drop rank   precision@K=0.556  recall@K=0.076

stale-visible rule fires on only 6 of 12,304 eligible pages
           precision among those 6: 0.667  (recall 0.001) -- blind to almost all of the risk


## 4. The unit of analysis, as a real dataframe

**One row = one (content page × decision date).** On the warehouse (ML-04) decision dates are monthly snapshots across the panel. The **starter CSV has exactly one implicit decision date** (export time), so today's dataframe is one row per page — a single slice of the eventual panel, enough to fix the schema and eligibility rule now.

**Eligibility filter** (from the ML-02 scaffolding), applied live below:

| Filter | Why |
|---|---|
| `impressions_prev_30d >= 100` | volume floor — kills the "game the label on a 60→30 page" failure |
| `content_age_days >= 90` | not brand-new (already true for every starter row; stated anyway) |
| `avg_position > 0` **or** `impressions_90d > 0` | drop pages with no search footprint at all |
| exclude `trend_direction == "down"` with `trend_pct <= -50` | pages already in freefall are the *recovery* problem — out of scope. Using `trend_direction` **here as a one-time row filter is not the same as using it as a feature** — it never enters the model. |

IDs shown below are pseudonyms (grouping/splitting only, never features); no client names or URLs exist in this data.

In [4]:
n0 = len(df)
drop_low_vol = int((~(df["impressions_prev_30d"] >= 100)).sum())
drop_new = int((~(df["content_age_days"] >= 90)).sum())
drop_no_search = int((~((df["avg_position"] > 0) | (df["impressions_90d"] > 0))).sum())
drop_freefall = int((freefall & elig).sum())

print(f"all pages:                       {n0:,}")
print(f"  - below 100 impr / prev 30d:   -{drop_low_vol:,}")
print(f"  - brand-new (<90d old):        -{drop_new:,}")
print(f"  - no search footprint:         -{drop_no_search:,}")
print(f"  - already in freefall:         -{drop_freefall:,}")
print(f"eligible (page x decision-date): {len(e):,}   across {e['client_id'].nunique()} clients")

cols = ["content_id", "client_id", "impressions_prev_30d", "clicks_prev_30d",
        "sessions_prev_30d", "avg_position", "days_since_last_update", "proxy_decline"]
e[cols].head(8)

all pages:                       30,000
  - below 100 impr / prev 30d:   -11,990
  - brand-new (<90d old):        -0
  - no search footprint:         -0
  - already in freefall:         -5,706
eligible (page x decision-date): 12,304   across 29 clients


,content_id,client_id,impressions_prev_30d,clicks_prev_30d,sessions_prev_30d,avg_position,days_since_last_update,proxy_decline
0,content_304f48230142,client_f369cb89fc,987,13,9,10.6,20,1
3,content_331d6c4de07b,client_19581e27de,4206,17,26,6.2,22,0
4,content_d99b7a2d90ca,client_3fdba35f04,6452,2,9,44.0,14,1
5,content_d4084a4bc775,client_f369cb89fc,1009,1,1,8.5,20,1
7,content_a63219c6e95a,client_19581e27de,632,0,4,21.2,22,0
9,content_c27558df2b0c,client_19581e27de,356,0,0,4.9,104,1
10,content_d8ee6cc6d642,client_19581e27de,6441,117,136,2.2,104,0
12,content_42fb2cad9ecf,client_6208ef0f77,1357,29,43,5.6,104,0


## 5. Why ML beats a fixed rule here

- **The current rule is late and narrow.** `trend_direction == "down"` only fires *after* a decline is already a measured 30-day fact; the stale-visible rule fires on a tiny fraction of eligible pages (see section 3 — a few dozen of ~12k) and misses declines that happen on fresh or lower-volume pages.
- **The signal is an interaction, not a threshold.** Near-future decline in this portfolio depends on position drift, query-mix narrowing, freshness, seasonality, traffic shape *and* volume together. The cross-tab below shows `proxy_decline` rate varies with both position tier and freshness tier, and no single horizontal or vertical cut separates high-risk from low-risk cells cleanly.
- **It drifts.** Over the warehouse's ~17 months, fixed thresholds calibrated on one quarter go stale; a model refit per fold tracks the shift. A learned ranking can also weight signals against each other and adapt per client, which an if-statement cannot.

A dashboard shows *what happened*; this predicts *what's about to* — that is the gap ML fills.

In [5]:
ct = pd.crosstab(
    e["position_tier"], e["freshness_tier"],
    values=e["proxy_decline"], aggfunc="mean",
).round(2)
print("proxy_decline rate by position_tier x freshness_tier (eligible pages):\n")
print(ct.to_string())
print(f"\noverall base rate: {e['proxy_decline'].mean():.2f}")
print("No single row or column threshold isolates the high-risk cells -> a fixed rule leaves signal on the table.")

proxy_decline rate by position_tier x freshness_tier (eligible pages):

freshness_tier  0-30  181+  31-90  91-180
position_tier                            
deep            0.14   NaN    NaN    0.19
page_1          0.35   0.5   0.12    0.40
page_3_5        0.35   0.5   0.20    0.37
striking        0.39   1.0   0.30    0.42
top_3           0.41   NaN   1.00    0.35

overall base rate: 0.37
No single row or column threshold isolates the high-risk cells -> a fixed rule leaves signal on the table.


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] `trend_direction` / `trend_pct` / `is_declining_label` appear only as named leakage or a one-time row filter — never as features
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.